# Mission 1 v2 · 음성 성별 분류 실험

**이 ipynb 한 파일에서 데이터 준비 → 학습 → 결과 표 → 혼동행렬(confusion matrix) → 오류 분석까지 실행합니다.**
**파일 버전: `2026-09-18-linux-r5-raw-counts`** — 첫 실행 셀에 같은 버전이 출력되는지 확인하세요.
필요한 코드를 아래 셀에 모두 넣어 별도 `src/` 폴더나 코드 ZIP 없이 사용할 수 있습니다.
원본 데이터와 Python 패키지는 별도로 필요합니다. 제출 파일 생성은 이 실험 흐름에서 제외했습니다.

기존 MFCC/F0 + LogisticRegression의 과거 Accuracy는 **94.34%**입니다.
새 모델의 점수는 실제 학습 후 아래 출력에서 확인합니다. **이 배포본에는 전체 WavLM 학습 결과가 아직 없습니다.**
CUDA GPU를 자동 사용하며 MPS/CPU도 지원합니다.
Colab에서는 런타임 유형을 GPU로 설정하고 아래 설정 셀의 `Device: cuda`를 확인하세요.

모델은 **WavLM-base-plus + layer 혼합 + attentive mean/std pooling**입니다.
무작위 6초 구간으로 학습하고 평가 때 최대 5개 구간의 확률을 평균합니다.
입력은 음성과 `startAt/endAt/speaker`, 학습 목표는 `gender`입니다.
서울 Training에서 fit/dev를 분리하고 공식 Validation은 모델 선택이 끝난 뒤 평가합니다.
전사·증상·주소는 모델 입력으로 사용하지 않습니다.

## 1. 환경 준비

패키지가 없는 환경이면 `INSTALL_DEPENDENCIES=True`로 설정하세요. uv로 **현재 커널**에 설치합니다.
이미 import한 패키지의 버전을 바꿨다면 커널을 재시작하세요. GPU 서버에서는 드라이버에 맞는
PyTorch wheel을 사용하며 CPU 버전은 강제하지 않습니다.

uv가 없으면 [공식 설치 안내](https://docs.astral.sh/uv/getting-started/installation/)를 따르세요.
Colab 셀에서는 `!curl -LsSf https://astral.sh/uv/install.sh | sh`로 설치할 수 있습니다.

In [1]:
from pathlib import Path
import os, sys, shutil, subprocess

NOTEBOOK_VERSION = '2026-09-18-linux-r5-raw-counts'
print('M1 v2 notebook:', NOTEBOOK_VERSION)
print('Kernel:', sys.executable)
INSTALL_DEPENDENCIES = False
PACKAGES = ['torch>=2.6,<3', 'transformers==4.57.6', 'numpy>=1.26,<3',
            'scipy>=1.14,<2', 'pandas>=2.2,<3', 'scikit-learn>=1.5,<2',
            'tqdm>=4.66', 'matplotlib>=3.9']
if INSTALL_DEPENDENCIES:
    uv = shutil.which('uv')
    if uv is None:
        candidate = Path.home()/'.local/bin/uv'
        uv = str(candidate) if candidate.is_file() else None
    if uv is None:
        raise RuntimeError('위 uv 설치 안내를 실행한 뒤 이 셀을 다시 실행하세요.')
    subprocess.run([uv, 'pip', 'install', '--python', sys.executable, *PACKAGES], check=True)
print('환경 준비 셀 완료. 다음 셀부터 순서대로 실행하세요.')

M1 v2 notebook: 2026-09-18-linux-r5-raw-counts
Kernel: /home/a202355692/micromamba/envs/aienv/bin/python
환경 준비 셀 완료. 다음 셀부터 순서대로 실행하세요.


In [2]:
import hashlib
import json
import math
import wave
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.signal import resample_poly
from sklearn.model_selection import StratifiedGroupKFold
from tqdm.auto import tqdm
import torch
from torch import nn
from transformers import WavLMConfig, WavLMModel
import contextlib
from dataclasses import asdict, dataclass
import importlib.metadata
import random
import shutil
import subprocess
import sys
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, roc_auc_score
from torch.utils.data import DataLoader, Dataset
from transformers import WavLMConfig, get_cosine_schedule_with_warmup
import os
import time
import argparse
import io
import zipfile
import csv
import matplotlib.pyplot as plt
from datetime import datetime
from IPython.display import display
from sklearn.metrics import (ConfusionMatrixDisplay, classification_report,
                             roc_curve, precision_recall_curve, average_precision_score)

SR = 16000
PREPROCESS_VERSION = 1
BUNDLE_VERSION = 1
BUNDLE_SR = 8000
print('PyTorch:', torch.__version__, '| Transformers:', importlib.metadata.version('transformers'))

PyTorch: 2.6.0+cu124 | Transformers: 4.57.6


## 2. 학습 설정

기본값은 전체 Training으로 8 epoch 학습입니다. 처음 연결만 점검하려면 `SMOKE_CALLS=100`으로 바꾸세요.
이는 학습 표본만 줄이며 모델 크기는 그대로입니다. CPU 전체 미세조정은 오래 걸릴 수 있습니다.
기본 effective batch는 4 × 누적 8 = 32입니다. GPU 메모리가 부족하면 batch_size=2,
accumulation=16을 사용합니다. 첫 1 epoch는 head만, 이후에는 backbone을 함께 학습합니다.

In [3]:
@dataclass
class TrainConfig:
    model_name: str = 'microsoft/wavlm-base-plus'
    seed: int = 42
    epochs: int = 8
    head_epochs: int = 1
    batch_size: int = 4
    accumulation: int = 8
    eval_batch_size: int = 8
    crop_seconds: float = 6.
    eval_windows: int = 5
    backbone_lr: float = 1e-5
    head_lr: float = 1e-4
    weight_decay: float = .01
    patience: int = 3
    num_workers: int = 0  # portable Windows/Jupyter default; Linux can use 2-4
    gradient_checkpointing: bool = True
    device: str = 'auto'


def choose_device(requested='auto'):
    if requested != 'auto':
        return torch.device(requested)
    if torch.cuda.is_available():
        return torch.device('cuda')
    if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def amp_context(device):
    if device.type != 'cuda':
        return contextlib.nullcontext()
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    return torch.autocast('cuda', dtype=dtype)

In [4]:
CFG = TrainConfig()
SMOKE_CALLS = None        # 연결 점검: 100, 전체 학습: None
RUN_VALIDATION = True    # 내부 dev로 설정을 확정한 뒤 공식 Validation 평가
RESUME_RUN_DIR = None   # 중단 재개 시 기존 실행 폴더 Path(...); 완료 epoch부터 재개

print('Device:', choose_device(CFG.device))
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name())
display(pd.Series(asdict(CFG), name='configuration'))

Device: cuda


GPU: NVIDIA A100 80GB PCIe


model_name                microsoft/wavlm-base-plus
seed                                             42
epochs                                            8
head_epochs                                       1
batch_size                                        4
accumulation                                      8
eval_batch_size                                   8
crop_seconds                                    6.0
eval_windows                                      5
backbone_lr                                 0.00001
head_lr                                      0.0001
weight_decay                                   0.01
patience                                          3
num_workers                                       0
gradient_checkpointing                         True
device                                         auto
Name: configuration, dtype: object

## 3. 원본 데이터와 결과 위치

**Colab GPU에서는 먼저 로컬 원본으로 만든 M1 전용 음성 번들을 Drive에 복사합니다.**
이 번들에는 전체 Training/Validation의 신고자 음성과 F/M 정답만 들어갑니다.
기존 `m1_audio_train.npz`(4,000통)와 `m1_audio_val.npz`(1,200통)는 전체 평가용이 아닙니다.
PC에 있는 파일은 노트북 업로드만으로 옮겨지지 않습니다.
`m3_text.json.gz`와 `src.zip`만으로도 M1 음성 학습을 할 수 없습니다.
로컬 커널이면 `DATA_ROOT = Path(r'C:\Users\doyun\DCC\대학부 데이터')`를 사용할 수 있습니다.
학교 Linux 서버에서는 `~/data/대학부 데이터`를 자동으로 찾고, 학습 전 WAV·JSON 네 폴더의
파일 수를 로컬 원본(Training 각 29,200개, Validation 각 3,640개)과 비교합니다.
이 Windows 경로는 Colab 원격 런타임에서 사용할 수 없습니다.
로컬에서 번들을 준비하는 명령:

```powershell
uv run --no-project --with numpy --with pandas --with scipy --with scikit-learn --with tqdm python -m src.m1_v2.bundle --output outputs/m1_v2_audio_full
```

완료된 `outputs/m1_v2_audio_full/` 폴더를 `MyDrive/dcc/m1_v2_audio_full/`로 옮기면
Colab 경로 셀은 이 번들을 직접 사용합니다. Drive에 원본 WAV/JSON이 이미 있다면 `DATA_ROOT`에
**Training과 Validation을 포함하는 상위 폴더**를 지정해도 됩니다.
로컬에서는 원본 데이터 폴더를 탐색합니다. 번들 ZIP은 노트북이 자동으로 풀므로 수동으로 풀지 마세요.
PC의 Drive 폴더에 복사가 끝나도 서버 동기화는 진행 중일 수 있습니다.
Drive에 `m1_v2_audio_full/extracted_audio/extracted_config.json`이 있으면
압축 해제된 전체 음성 파일을 확인한 뒤 바로 사용합니다. 그렇지 않으면 ZIP을 Colab 디스크로 준비합니다.

```text
대학부 데이터/
  Training/1.원천데이터/TS_서울_구급/*.wav
  Training/2.라벨링데이터/TL_서울_구급/*.json
  Validation/1.원천데이터/VS_서울_구급/*.wav
  Validation/2.라벨링데이터/VL_서울_구급/*.json
```

실험 폴더에는 학습 이력, 체크포인트, 예측 표, 그림이 저장됩니다. 원본은 수정하지 않습니다.
먼저 아래 진단 셀이 Drive 연결과 `dcc` 폴더 내용을 표시합니다. 오디오가 없으면 경로 이름만 바꿔도 해결되지 않습니다.

In [5]:
def dataset_layout(root):
    root = Path(root).expanduser().resolve()
    return {
        'TRAIN_AUDIO': root/'Training/1.원천데이터/TS_서울_구급',
        'TRAIN_LABEL': root/'Training/2.라벨링데이터/TL_서울_구급',
        'VAL_AUDIO': root/'Validation/1.원천데이터/VS_서울_구급',
        'VAL_LABEL': root/'Validation/2.라벨링데이터/VL_서울_구급',
    }


def inspect_dataset(root, require_validation=True):
    layout = dataset_layout(root)
    keys = list(layout) if require_validation else ['TRAIN_AUDIO', 'TRAIN_LABEL']
    status = []
    for key in keys:
        path = layout[key]
        extension = '*.wav' if key.endswith('AUDIO') else '*.json'
        try:
            exists = path.is_dir()
            has_files = exists and next(path.glob(extension), None) is not None
        except OSError:
            exists = has_files = False
        status.append({'folder': key, 'path': str(path), 'exists': exists,
                       'has_required_files': has_files, 'required_type': extension})
    return status


def compare_dataset_counts(root):
    """Count original files in each split before training; fail on partial uploads."""
    expected = {'TRAIN_AUDIO': 29200, 'TRAIN_LABEL': 29200,
                'VAL_AUDIO': 3640, 'VAL_LABEL': 3640}
    rows = []
    for key, folder in dataset_layout(root).items():
        suffix = '.wav' if key.endswith('AUDIO') else '.json'
        if not folder.is_dir():
            actual = 0
        else:
            with os.scandir(folder) as entries:
                actual = sum(entry.is_file() and entry.name.lower().endswith(suffix)
                             for entry in entries)
        rows.append({'folder': key, 'expected': expected[key], 'actual': actual,
                     'match': actual == expected[key], 'path': str(folder)})
    return rows


def find_data_roots(search_roots, require_validation=True, max_depth=4,
                    max_dirs=1500, max_seconds=20):
    """Inspect directory names only; prune source-data leaves and caches.

    Returns matches plus whether a scan budget was reached. A partial scan never
    establishes that no data exists elsewhere. Multiple matches require a choice.
    """
    from collections import deque
    skip = {'Training', 'Validation', 'cache', 'ckpt', 'runs', 'outputs', 'artifacts',
            'data_label', 'node_modules', '.git', '__pycache__', 'm3_out'}
    queue = deque((Path(p).expanduser().resolve(), 0) for p in search_roots)
    seen, matches = set(), []
    started = time.monotonic()
    while queue and len(seen) < max_dirs and time.monotonic()-started < max_seconds:
        root, depth = queue.popleft()
        if root in seen or not root.is_dir():
            continue
        seen.add(root)
        # Cheap directory existence check before looking for files in real leaf folders.
        if (root/'Training').is_dir():
            status = inspect_dataset(root, require_validation)
            if all(row['has_required_files'] for row in status):
                matches.append(root)
                continue
        if depth >= max_depth:
            continue
        try:
            with os.scandir(root) as entries:
                for entry in entries:
                    if entry.name in skip or entry.name.startswith('.') or entry.name == 'drive':
                        continue
                    if entry.is_dir(follow_symlinks=False):
                        queue.append((Path(entry.path), depth+1))
        except OSError:
            continue
    return sorted(set(matches)), bool(queue)


def resolve_data_root(explicit_root, quick_candidates, search_roots, require_validation=True):
    if explicit_root is not None and str(explicit_root).strip():
        root = Path(explicit_root).expanduser().resolve()
        status = inspect_dataset(root, require_validation)
        if not all(row['has_required_files'] for row in status):
            details = '\n'.join(f"  {row['folder']}: {row['path']} [{row['required_type']} 없음]"
                                for row in status if not row['has_required_files'])
            raise FileNotFoundError('지정한 DATA_ROOT의 원본 데이터가 없거나 비어 있습니다.\n'+details)
        return root
    matches = sorted({Path(p).expanduser().resolve() for p in quick_candidates
                      if (Path(p)/'Training').is_dir()
                      and all(row['has_required_files'] for row in inspect_dataset(p, require_validation))})
    limited = False
    if not matches:
        matches, limited = find_data_roots(search_roots, require_validation)
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        raise ValueError('데이터 후보가 여러 개입니다. DATA_ROOT에 사용할 경로 하나를 지정하세요:\n'
                         + '\n'.join(str(p) for p in matches))
    searched = '\n'.join('  '+str(p) for p in search_roots)
    suffix = '\n탐색 제한에 도달했습니다. 더 깊은 경로는 DATA_ROOT로 직접 지정하세요.' if limited else ''
    raise FileNotFoundError(
        '현재 런타임에서 M1 원본 WAV+JSON을 찾지 못했습니다.\n'
        'Colab은 PC의 C:/Users/... 경로를 직접 읽을 수 없습니다.\n'
        'm3_text.json.gz와 src.zip에는 M1 학습용 음성이 없습니다.\n'
        '1) 원본이 Drive에 있으면 DATA_ROOT를 Training/Validation의 상위 폴더로 지정하세요.\n'
        '2) ZIP만 있으면 Training/Validation 구조로 압축을 풀어야 합니다.\n'
        '3) PC에만 있다면 원본 WAV와 JSON을 Drive로 옮기거나 로컬 커널에서 실행하세요.\n'
        '탐색 위치:\n'+searched+suffix)

In [6]:
def wait_for_bundle(bundle_dir, timeout_seconds=1200, poll_seconds=15):
    """Wait for every expected shard to become visible; never train on a partial transfer."""
    if timeout_seconds < 0 or poll_seconds <= 0:
        raise ValueError('Require timeout >= 0 and poll interval > 0')
    bundle_dir = Path(bundle_dir)
    config_path = bundle_dir/'bundle_config.json'
    if not config_path.is_file():
        raise FileNotFoundError(f'{config_path}이 보이지 않습니다. Drive 계정과 번들 경로를 확인하세요.')
    config = json.loads(config_path.read_text(encoding='utf-8'))
    if config.get('version') != BUNDLE_VERSION or not config.get('complete'):
        raise ValueError('Unsupported or incomplete M1 bundle configuration')
    expected = {}
    for split, key in [('train', 'training_calls'), ('validation', 'validation_calls')]:
        count = int(config[key])
        if count <= 0:
            raise ValueError(f'Invalid {key}: {count}')
        expected[split] = {f'{split}_{i:03d}.zip' for i in range((count+499)//500)}
    start = time.monotonic()
    while True:
        missing, progress = [], []
        for split, names in expected.items():
            visible = {p.name for p in bundle_dir.glob(f'{split}_???.zip')}
            extra = visible - names
            if extra:
                raise ValueError(f'예상 외 {split} ZIP: {sorted(extra)}. 번들 폴더를 확인하세요.')
            missing.extend(sorted(names-visible))
            progress.append(f'{split} ZIP: {len(visible)}/{len(names)}')
        elapsed = time.monotonic()-start
        print(' | '.join(progress), f'| 대기 {int(elapsed)}초', flush=True)
        if not missing:
            print('모든 ZIP이 현재 런타임에서 보입니다. 압축 해제를 시작합니다.', flush=True)
            return config
        remaining = timeout_seconds-elapsed
        if remaining <= 0:
            raise TimeoutError(
                f'Drive 동기화 대기 시간이 끝났습니다. 아직 {len(missing)}개가 보이지 않습니다: '
                f'{missing[:10]}. PC의 Google Drive 앱에서 동기화 일시중지·오류·용량을 확인한 뒤 '
                '이 셀만 재실행하세요. 불완전한 데이터로 학습하지 않았습니다.')
        print(f'아직 안 보이는 파일 {len(missing)}개: {missing[:5]}. '
              f'{min(poll_seconds, remaining):.0f}초 후 다시 확인합니다. 중단하려면 셀 실행을 정지하세요.',
              flush=True)
        time.sleep(min(poll_seconds, remaining))


def load_bundle(bundle_dir, extract_dir, workers=4):
    """Extract a completed transfer to local runtime disk; return both manifests."""
    bundle_dir, extract_dir = Path(bundle_dir), Path(extract_dir)
    extract_dir = extract_dir.expanduser().resolve()
    config_path = bundle_dir/'bundle_config.json'
    if not config_path.is_file():
        raise FileNotFoundError(f'Incomplete M1 transfer: {config_path}')
    config = json.loads(config_path.read_text(encoding='utf-8'))
    if config.get('version') != BUNDLE_VERSION or config.get('sr') != BUNDLE_SR or not config.get('complete'):
        raise ValueError('Unsupported or incomplete M1 bundle')
    extract_dir.mkdir(parents=True, exist_ok=True)
    frames = {}
    for split, count_key in [('train', 'training_calls'), ('validation', 'validation_calls')]:
        shards = sorted(bundle_dir.glob(f'{split}_???.zip'))
        if not shards:
            raise FileNotFoundError(f'No {split} shard ZIPs in {bundle_dir}')
        rows = []
        split_dir = extract_dir/split
        split_dir.mkdir(parents=True, exist_ok=True)
        for shard in tqdm(shards, desc=f'Extract {split}'):
            with zipfile.ZipFile(shard) as z:
                manifest = pd.read_csv(io.BytesIO(z.read('manifest.csv')),
                                       dtype={'stem': str, 'audio_file': str})
                names = {f'audio/{stem}.npy' for stem in manifest.stem}
                if not names.issubset(set(z.namelist())):
                    raise ValueError(f'Missing audio entry in {shard}')
                def extract_one(stem):
                    member = f'audio/{stem}.npy'
                    out = split_dir/f'{stem}.npy'
                    if not out.is_file() or out.stat().st_size != z.getinfo(member).file_size:
                        temporary = out.with_suffix('.tmp')
                        with z.open(member) as src, temporary.open('wb') as dst:
                            import shutil
                            shutil.copyfileobj(src, dst)
                        temporary.replace(out)
                with ThreadPoolExecutor(max_workers=workers) as pool:
                    list(pool.map(extract_one, manifest.stem))
                manifest['path'] = [str(split_dir/f'{stem}.npy')
                                    for stem in manifest.stem]
                rows.append(manifest)
        frames[split] = pd.concat(rows, ignore_index=True)
        if len(frames[split]) != config[count_key] or frames[split].stem.duplicated().any():
            raise ValueError(f'Incomplete or duplicate {split} bundle')
    if config['limit'] is None and (config['training_calls'] != 29200 or config['validation_calls'] != 3640):
        raise ValueError('Full bundle must contain all 29,200/3,640 calls')
    return frames['train'], frames['validation'], config


def load_extracted_bundle(extract_dir):
    """Load unpacked Drive/local audio, rebasing paths for the current runtime."""
    extract_dir = Path(extract_dir).expanduser().resolve()
    config = json.loads((extract_dir/'extracted_config.json').read_text(encoding='utf-8'))
    if (config.get('version') != BUNDLE_VERSION or config.get('sr') != BUNDLE_SR
            or config.get('storage') != 'npy_per_call' or not config.get('complete')):
        raise ValueError('Unsupported extracted M1 bundle')
    frames = {}
    for split, key in [('train', 'training_calls'), ('validation', 'validation_calls')]:
        frame = pd.read_csv(extract_dir/f'{split}_manifest.csv', dtype={'stem': str, 'audio_file': str})
        if len(frame) != config[key] or frame.stem.duplicated().any():
            raise ValueError(f'Incomplete {split} manifest')
        paths = [extract_dir/split/f'{stem}.npy' for stem in frame.stem]
        visible = {p.name for p in (extract_dir/split).glob('*.npy')}
        missing = [p.name for p in paths if p.name not in visible]
        if missing:
            raise FileNotFoundError(f'압축 해제된 {split} 음성 {len(missing)}개가 아직 보이지 않습니다: {missing[:5]}')
        frame['path'] = [str(p) for p in paths]
        frames[split] = frame
    if config['limit'] is None and (len(frames['train']) != 29200 or len(frames['validation']) != 3640):
        raise ValueError('Full extracted bundle must contain all 29,200/3,640 calls')
    return frames['train'], frames['validation'], config

In [7]:
def expected_m1_shards(config):
    if (config.get('version') != 1 or config.get('sr') != 8000
            or not config.get('complete') or config.get('label_map') != {'F': 0, 'M': 1}):
        raise ValueError('Unsupported M1 bundle configuration')
    counts = [int(config['training_calls']), int(config['validation_calls'])]
    if min(counts) <= 0:
        raise ValueError('M1 bundle call counts must be positive')
    if config.get('limit') is None and counts != [29200, 3640]:
        raise ValueError('전체 번들은 Training 29,200통 / Validation 3,640통이어야 합니다.')
    expected = {}
    for split, count in zip(('train', 'validation'), counts):
        for start in range(0, count, 500):
            expected[f'{split}_{start//500:03d}.zip'] = min(500, count-start)
    return expected


def check_m1_shard(path, expected_calls):
    """Check the manifest and ZIP directory; extraction later also verifies payload CRCs."""
    with zipfile.ZipFile(path) as archive:
        rows = list(csv.DictReader(io.StringIO(archive.read('manifest.csv').decode('utf-8'))))
        if len(rows) != expected_calls:
            raise ValueError(f'{path.name}: manifest call count mismatch')
        stems = [row['stem'] for row in rows]
        if len(set(stems)) != len(stems) or any('/' in s or '\\' in s or s in ('.', '..') for s in stems):
            raise ValueError(f'{path.name}: invalid or duplicate call IDs')
        audio = {f'audio/{stem}.npy' for stem in stems}
        actual = [name for name in archive.namelist() if name.startswith('audio/')]
        if set(actual) != audio or len(actual) != len(audio):
            raise ValueError(f'{path.name}: audio members do not match manifest')


def stage_m1_bundle(source_dirs, cache_dir, upload_fn=None):
    """Merge available shards into runtime-local storage; request only missing files.

    upload_fn(missing_names) must place user-selected files in one of source_dirs and
    return their names. Empty selection cancels. Training never receives a partial bundle.
    """
    cache_dir = Path(cache_dir).expanduser().resolve()
    sources = list(dict.fromkeys(Path(p).expanduser().resolve() for p in source_dirs if p is not None))
    cache_dir.mkdir(parents=True, exist_ok=True)
    marker = cache_dir/'.transfer_config.json'

    def find_config():
        return next((p for p in [*(d/'bundle_config.json' for d in sources),
                                  cache_dir/'bundle_config.json', marker] if p.is_file()), None)

    config_path = find_config()
    if config_path is None and upload_fn is not None:
        upload_fn(['bundle_config.json'])
        config_path = find_config()
    if config_path is None:
        raise FileNotFoundError('bundle_config.json이 없습니다. 번들 경로를 지정하거나 이 파일을 직접 업로드하세요.')
    config = json.loads(config_path.read_text(encoding='utf-8'))
    expected = expected_m1_shards(config)
    for existing in (marker, cache_dir/'bundle_config.json'):
        if existing.is_file() and json.loads(existing.read_text(encoding='utf-8')) != config:
            raise ValueError('캐시에 다른 번들의 설정이 있습니다. CACHE_ROOT를 새 폴더로 지정하세요.')
    marker.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding='utf-8')
    print('[data] configuration:', config_path, flush=True)
    ready = set()
    previous_missing = None
    while True:
        missing = []
        for name, count in expected.items():
            if name in ready:
                continue
            target = cache_dir/name
            candidates = [target, *(d/name for d in sources if d != cache_dir)]
            for candidate in candidates:
                if not candidate.is_file():
                    continue
                try:
                    check_m1_shard(candidate, count)
                    if candidate != target:
                        temporary = target.with_suffix('.part')
                        try:
                            shutil.copyfile(candidate, temporary)
                            check_m1_shard(temporary, count)
                            temporary.replace(target)
                        finally:
                            temporary.unlink(missing_ok=True)
                    ready.add(name)
                    print(f'[cache] {len(ready)}/{len(expected)} {name}', flush=True)
                    break
                except (OSError, ValueError, KeyError, zipfile.BadZipFile) as exc:
                    print(f'[data] {name} 읽기 실패: {type(exc).__name__}: {exc}', flush=True)
            if name not in ready:
                missing.append(name)
        if not missing:
            final = cache_dir/'bundle_config.json'
            temporary = final.with_suffix('.tmp')
            temporary.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding='utf-8')
            temporary.replace(final)
            print(f'[ready] ZIP {len(expected)}개를 런타임 디스크에서 확인했습니다.', flush=True)
            return cache_dir, config
        print(f'[missing] {len(missing)}개: '+', '.join(missing), flush=True)
        if upload_fn is None:
            raise FileNotFoundError('미완료 번들: '+', '.join(missing)+
                                    '. 동기화 후 셀을 재실행하거나 ALLOW_DIRECT_UPLOAD=True로 설정하세요.')
        if previous_missing == missing:
            raise FileNotFoundError('직접 업로드 후에도 필요한 ZIP이 확인되지 않았습니다. '+
                                    '누락 목록과 파일 이름을 확인하고 이 셀을 재실행하세요.')
        previous_missing = missing
        uploaded = upload_fn(missing)
        if not uploaded:
            raise FileNotFoundError('직접 업로드가 취소됐습니다. 준비된 캐시는 유지됩니다. '+
                                    '파일을 준비한 후 이 셀을 다시 실행하세요.')

### Google Drive 연결

이미 연결된 Drive는 그대로 사용합니다. 인증을 다시 시도하려면
`FORCE_DRIVE_REMOUNT=True`로 바꾸고 **이 연결 셀만** 실행하세요.
인증 창에서는 번들이 저장된 Google 계정을 선택합니다. M3 v4처럼 연결을 두 번 시도하고,
실패하면 Colab 디스크의 데이터와 직접 업로드를 사용할 수 있도록 진행합니다.

`credential-propagation` 오류는 Google 인증 단계의 실패입니다. 일반 브라우저에서
Colab을 직접 열고 계정·로그인 팝업·쿠키 차단 설정을 확인한 뒤 재시도하세요.
이 셀의 코드는 브라우저의 사용자 인증을 대신할 수 없습니다.

In [8]:
def connect_colab_drive(force_remount=False):
    from google.colab import drive
    root = Path('/content/drive/MyDrive')
    if not force_remount and os.path.ismount('/content/drive') and root.is_dir():
        print('기존 Google Drive 연결을 사용합니다:', root)
        return root
    attempts = [True] if force_remount else [False, True]
    for attempt, force in enumerate(attempts, 1):
        try:
            drive.mount('/content/drive', force_remount=force)
            if root.is_dir():
                print('Google Drive 연결 완료:', root)
                return root
        except Exception as exc:
            print(f'[drive] 시도 {attempt} 실패: {type(exc).__name__}: {exc}', flush=True)
    print('[drive] 연결되지 않았습니다. 다음 셀에서 로컬 캐시 또는 직접 업로드를 사용합니다.')
    return None

IN_COLAB = 'google.colab' in sys.modules or 'COLAB_RELEASE_TAG' in os.environ
FORCE_DRIVE_REMOUNT = False
DRIVE_OK = False
if IN_COLAB:
    DRIVE_OK = connect_colab_drive(force_remount=FORCE_DRIVE_REMOUNT) is not None
else:
    print('로컬 커널: Google Drive 인증을 건너뜁니다.')

로컬 커널: Google Drive 인증을 건너뜁니다.


### M3 v4 방식의 데이터 복구

이미 압축 해제된 `extracted_audio`가 완성돼 있으면 이를 우선 사용합니다.
그 외에는 Drive의 ZIP을 Colab 디스크로 복사하고, 일부만 보이면 **누락된 파일만** 직접 업로드할 수 있습니다.
선택할 파일은 PC의 `C:/Users/doyun/DCC/outputs/m1_v2_audio_full/`에 있습니다.
M3의 텍스트 파일은 작지만 M1 음성은 크므로, 업로드 창에서는 **ZIP 하나씩** 선택하세요.
이미 준비한 ZIP은 캐시에 남아 다음 실행에 재사용됩니다. 업로드 대신 동기화를 기다리려면
`ALLOW_DIRECT_UPLOAD=False`로 설정하고 동기화 완료 후 이 셀을 재실행하세요.
Drive를 사용하지 못한 경우 결과는 Colab 임시 디스크에 저장되므로 세션 종료 전에 내려받아야 합니다.

In [9]:
def upload_missing_m1(names, upload_dir):
    from google.colab import files
    upload_dir = Path(upload_dir)
    upload_dir.mkdir(parents=True, exist_ok=True)
    print('[upload] 필요한 파일:', ', '.join(names), flush=True)
    print('[upload] PC의 C:/Users/doyun/DCC/outputs/m1_v2_audio_full/에서 선택하세요.')
    print('[upload] 음성 ZIP은 메모리 사용을 줄이기 위해 한 번에 하나씩 선택하세요.')
    previous_dir = Path.cwd()
    try:
        os.chdir(upload_dir)
        uploaded = files.upload()
        selected = list(uploaded)
        del uploaded
    finally:
        os.chdir(previous_dir)
    print('[upload] 받은 파일:', selected)
    return selected

In [10]:
IN_COLAB = 'google.colab' in sys.modules or 'COLAB_RELEASE_TAG' in os.environ
server_data_root = Path.home()/'data/대학부 데이터'
DATA_ROOT = (server_data_root if not IN_COLAB and sys.platform.startswith('linux')
             and server_data_root.is_dir() else None)
# 다른 위치의 원본 WAV/JSON을 쓰려면 DATA_ROOT = Path('/실제/경로')로 지정
DATA_BUNDLE = (Path('/content/drive/MyDrive/dcc/m1_v2_audio_full')
               if IN_COLAB and DATA_ROOT is None else None)
ALLOW_DIRECT_UPLOAD = True  # M3 v4처럼 누락된 파일만 직접 업로드
# DATA_ROOT는 반드시 실제 파일이 있는 곳이어야 합니다. 임의로 폴더만 만들지 마세요.
print('Current directory:', Path.cwd())
drive_root = Path('/content/drive/MyDrive')
if IN_COLAB:
    print('Drive mounted:', drive_root.is_dir())
    dcc_dir = drive_root/'dcc'
    if dcc_dir.is_dir():
        print('Drive dcc contents:', [p.name for p in list(dcc_dir.iterdir())[:30]])
    if not drive_root.is_dir():
        print('[drive] 미연결: Colab 캐시/직접 업로드 경로를 사용합니다.')

bundle_training = bundle_validation = None
if DATA_BUNDLE is not None:
    DATA_BUNDLE = Path(DATA_BUNDLE).expanduser().resolve()
    CACHE_ROOT = Path('/content/m1_v2_cache') if IN_COLAB else Path.cwd()/'cache/m1_v2'
    extracted_candidate = DATA_BUNDLE/'extracted_audio'
    if (extracted_candidate/'extracted_config.json').is_file():
        try:
            bundle_training, bundle_validation, bundle_config = load_extracted_bundle(extracted_candidate)
            print('[data] 압축 해제된 음성을 직접 사용합니다:', extracted_candidate)
        except FileNotFoundError as exc:
            print('[data] 압축 해제 파일의 동기화가 미완료입니다:', exc)
            print('[data] ZIP/직접 업로드 경로로 확인합니다.')
    if bundle_training is not None:
        transfer_config = bundle_config
    elif IN_COLAB:
        upload_root = Path('/content/m1_v2_uploads')
        source_dirs = [upload_root, DATA_BUNDLE, Path('/content/m1_v2_audio_full'),
                       Path('/content'), Path.cwd()/'m1_v2_audio_full',
                       drive_root/'m1_v2_audio_full', drive_root/'dcc/outputs/m1_v2_audio_full',
                       drive_root/'2026-2학기/dcc/m1_v2_audio_full']
        if drive_root.is_dir():
            for pattern in ('*/m1_v2_audio_full/bundle_config.json',
                            '*/*/m1_v2_audio_full/bundle_config.json'):
                source_dirs.extend(p.parent for p in drive_root.glob(pattern))
        upload_fn = (lambda names: upload_missing_m1(names, upload_root)) if ALLOW_DIRECT_UPLOAD else None
        DATA_BUNDLE, transfer_config = stage_m1_bundle(
            source_dirs, CACHE_ROOT/'bundle', upload_fn=upload_fn)
    else:
        transfer_config = wait_for_bundle(DATA_BUNDLE, timeout_seconds=0)
    if transfer_config['limit'] is not None and SMOKE_CALLS is None:
        raise ValueError('이 번들은 일부 통화만 포함합니다. SMOKE_CALLS를 설정하거나 전체 번들을 준비하세요.')
    if bundle_training is None:
        bundle_training, bundle_validation, bundle_config = load_bundle(
            DATA_BUNDLE, CACHE_ROOT/'extracted_audio')
    if bundle_config['limit'] is None and SMOKE_CALLS is not None:
        bundle_training = bundle_training.iloc[:SMOKE_CALLS].reset_index(drop=True)
    RUNS_ROOT = ((drive_root/'dcc/runs/m1_v2') if drive_root.is_dir()
                 else Path('/content/runs/m1_v2')) if IN_COLAB else Path.cwd()/'runs/m1_v2'
    if IN_COLAB and not drive_root.is_dir():
        print('[out] Colab 임시 디스크에 결과를 저장합니다. 세션 종료 전에 내려받으세요.')
    TRAIN_AUDIO = TRAIN_LABEL = VAL_AUDIO = VAL_LABEL = None
    print('M1 data bundle:', DATA_BUNDLE)
    print('Calls:', len(bundle_training), len(bundle_validation))
else:
    candidates = []
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidates.extend([parent, parent/'대학부 데이터', parent/'data/raw'])
    search_roots = [Path.cwd()]
    if IN_COLAB:
        candidates.extend([drive_root/'dcc/대학부 데이터', drive_root/'대학부 데이터',
                           drive_root/'dcc', drive_root/'dcc/data/raw'])
        search_roots = [Path('/content'), drive_root]
        shared = Path('/content/drive/Shareddrives')
        if shared.is_dir():
            search_roots.append(shared)
    try:
        DATA_ROOT = resolve_data_root(DATA_ROOT, candidates, search_roots,
                                      require_validation=RUN_VALIDATION and SMOKE_CALLS is None)
    except FileNotFoundError as exc:
        if IN_COLAB:
            raise FileNotFoundError(
                'Colab Drive에서 M1 음성을 찾지 못했습니다. 로컬 PC의 '
                'outputs/m1_v2_audio_full 폴더 전체를 본인 Drive의 '
                'MyDrive/dcc/m1_v2_audio_full에 복사해야 합니다. '
                'bundle_config.json과 train_000.zip, validation_000.zip이 '
                '그 폴더에 보여야 합니다. 복사 후 이 경로 셀을 다시 실행하세요. '
                '다른 위치에 복사했다면 DATA_BUNDLE = Path(Drive의 실제 경로)로 지정하세요. '
                'PC의 C:/Users/... 경로는 Colab에서 직접 읽을 수 없습니다.'
            ) from exc
        raise
    RUNS_ROOT = DATA_ROOT.parent/'runs/m1_v2'
    CACHE_ROOT = Path('/content/m1_v2_cache') if IN_COLAB else DATA_ROOT.parent/'cache/m1_v2'
    layout = dataset_layout(DATA_ROOT)
    count_rows = compare_dataset_counts(DATA_ROOT)
    display(pd.DataFrame(count_rows)[['folder', 'expected', 'actual', 'match', 'path']])
    if not all(row['match'] for row in count_rows):
        raise ValueError('원본 데이터 파일 수가 로컬 전체 데이터와 다릅니다. 업로드를 확인하세요.')
    TRAIN_AUDIO, TRAIN_LABEL = layout['TRAIN_AUDIO'], layout['TRAIN_LABEL']
    VAL_AUDIO, VAL_LABEL = layout['VAL_AUDIO'], layout['VAL_LABEL']
    display(pd.DataFrame(inspect_dataset(DATA_ROOT,
        require_validation=RUN_VALIDATION and SMOKE_CALLS is None)))
    print('Data:', DATA_ROOT)
RUN_DIR = (Path(RESUME_RUN_DIR) if RESUME_RUN_DIR else
           RUNS_ROOT/datetime.now().strftime('%Y%m%d_%H%M%S_wavlm_experiment'))
print('Results:', RUN_DIR)

Current directory: /home/a202355692/contest/dcc


,folder,expected,actual,match,path
0,TRAIN_AUDIO,29200,29200,True,/home/a202355692/data/raw/Training/1.원천데이터/TS_...
1,TRAIN_LABEL,29200,29200,True,/home/a202355692/data/raw/Training/2.라벨링데이터/TL...
2,VAL_AUDIO,3640,3640,True,/home/a202355692/data/raw/Validation/1.원천데이터/V...
3,VAL_LABEL,3640,3640,True,/home/a202355692/data/raw/Validation/2.라벨링데이터/...


,folder,path,exists,has_required_files,required_type
0,TRAIN_AUDIO,/home/a202355692/data/raw/Training/1.원천데이터/TS_...,True,True,*.wav
1,TRAIN_LABEL,/home/a202355692/data/raw/Training/2.라벨링데이터/TL...,True,True,*.json
2,VAL_AUDIO,/home/a202355692/data/raw/Validation/1.원천데이터/V...,True,True,*.wav
3,VAL_LABEL,/home/a202355692/data/raw/Validation/2.라벨링데이터/...,True,True,*.json


Data: /home/a202355692/data/raw
Results: /home/a202355692/data/runs/m1_v2/20260919_165146_wavlm_experiment


## 4. 전처리 — 신고자 음성 추출과 내부 분할

신고자 구간을 합치고 대원과 겹치는 부분을 뺍니다. 8→16 kHz 변환은 모델 입력 규격을 맞추며
잃어버린 고주파를 복원하는 것은 아닙니다. 모든 통화에 같은 전처리를 적용합니다.
음성이 없거나 0.25초 미만/무음이면 fit 사전확률을 사용하고 평가 분모에도 남깁니다.

In [11]:
def merge_intervals(intervals):
    merged = []
    for start, end in sorted(intervals):
        if end <= start:
            continue
        if merged and start <= merged[-1][1]:
            merged[-1] = (merged[-1][0], max(end, merged[-1][1]))
        else:
            merged.append((start, end))
    return merged


def caller_intervals(utterances, duration_ms):
    """Union caller intervals, then subtract other speakers; milliseconds throughout."""
    caller, other = [], []
    for u in utterances:
        start, end = float(u['startAt']), float(u['endAt'])
        speaker = int(u['speaker'])
        if not math.isfinite(start) or not math.isfinite(end) or end < start:
            raise ValueError('Invalid utterance timestamps')
        if speaker not in (0, 1):
            raise ValueError(f'Unexpected speaker: {speaker}')
        interval = (max(0., start), min(duration_ms, end))
        (caller if speaker == 1 else other).append(interval)
    clean = merge_intervals(caller)
    for left, right in merge_intervals(other):
        next_clean = []
        for start, end in clean:
            if right <= start or left >= end:
                next_clean.append((start, end))
            else:
                if start < left:
                    next_clean.append((start, left))
                if right < end:
                    next_clean.append((right, end))
        clean = next_clean
    return clean


def extract_caller(wav_path, utterances):
    with wave.open(str(wav_path), 'rb') as wav:
        if wav.getsampwidth() != 2:
            raise ValueError(f'Expected 16-bit PCM: {wav_path}')
        rate, channels = wav.getframerate(), wav.getnchannels()
        intervals = caller_intervals(utterances, 1000 * wav.getnframes() / rate)
        chunks = []
        for start, end in intervals:
            i, j = int(start * rate / 1000), int(end * rate / 1000)
            if j <= i:
                continue
            wav.setpos(i)
            x = np.frombuffer(wav.readframes(j-i), dtype='<i2').reshape(-1, channels)
            chunks.append(x.astype(np.float32).mean(1) / 32768.)
    x = np.concatenate(chunks) if chunks else np.empty(0, dtype=np.float32)
    if len(x) and rate != SR:
        gcd = math.gcd(rate, SR)
        x = resample_poly(x, SR // gcd, rate // gcd).astype(np.float32)
    return x

동일 음성 해시가 fit/dev 양쪽에 들어가지 않도록 묶습니다. 실제 화자 ID 기반 분할은 아니므로
다른 녹음에 동일 인물이 있는 경우까지 보장하지는 않습니다. 누락·손상 파일은 조용히 제외하지 않습니다.

In [12]:
def prepare_manifest(audio_dir, label_dir, cache_dir, *, labeled=True, workers=4, limit=None):
    """One row per WAV, including empty caller audio. Bad/missing input fails loudly.

    Cache key uses allowed fields, WAV size/mtime and preprocessing version.
    `gender` is read only as the target when labeled=True; inference ignores it.
    """
    audio_dir, label_dir, cache_dir = map(Path, (audio_dir, label_dir, cache_dir))
    cache_dir.mkdir(parents=True, exist_ok=True)
    files = sorted(audio_dir.glob('*.wav'))
    if not files:
        raise FileNotFoundError(f'No WAV files in {audio_dir}')
    if limit is not None:
        files = files[:limit]  # smoke test only; full training uses every call

    def one(path):
        with (label_dir / f'{path.stem}.json').open(encoding='utf-8-sig') as f:
            raw = json.load(f)
        utts = [{k: u[k] for k in ('startAt', 'endAt', 'speaker')} for u in raw['utterances']]
        target = raw['gender'] if labeled else None
        if labeled and target not in ('F', 'M'):
            raise ValueError(f'Unknown gender target in {path.stem}')
        stat = path.stat()
        signature = [PREPROCESS_VERSION, str(path.resolve()), stat.st_size, stat.st_mtime_ns, utts]
        key = hashlib.sha256(json.dumps(signature, sort_keys=True).encode()).hexdigest()
        cached = cache_dir / f'{key}.npy'
        if not cached.exists():
            x = extract_caller(path, utts)
            temporary = cached.with_suffix('.tmp')
            with temporary.open('wb') as f:
                np.save(f, x, allow_pickle=False)
            temporary.replace(cached)
        x = np.load(cached, mmap_mode='r', allow_pickle=False)
        usable = len(x) >= SR // 4 and float(np.std(x)) > 1e-5
        return dict(stem=path.stem, audio_file=path.name, path=str(cached.resolve()),
                    y={'F': 0, 'M': 1}.get(target, -1), seconds=len(x)/SR,
                    usable=usable, audio_hash=hashlib.sha256(x.tobytes()).hexdigest())

    with ThreadPoolExecutor(max_workers=workers) as pool:
        rows = list(tqdm(pool.map(one, files), total=len(files), desc='Caller audio'))
    return pd.DataFrame(rows)


def split_training(frame, seed=42):
    """20% internal dev; identical extracted waveforms remain in the same fold."""
    if frame['stem'].duplicated().any() or set(frame.y) != {0, 1}:
        raise ValueError('Unique calls and both F/M targets are required')
    groups = frame.audio_hash.where(frame.usable, 'empty:' + frame.stem)
    for label in (0, 1):
        if groups[frame.y == label].nunique() < 5:
            raise ValueError('Need at least five distinct call groups per class')
    splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    fit, dev = next(splitter.split(frame, frame.y, groups))
    return frame.iloc[fit].reset_index(drop=True), frame.iloc[dev].reset_index(drop=True)


def assert_disjoint(train, validation):
    if set(train.stem) & set(validation.stem):
        raise ValueError('Call IDs overlap across splits')
    if set(train.loc[train.usable, 'audio_hash']) & set(validation.loc[validation.usable, 'audio_hash']):
        raise ValueError('Identical caller waveforms overlap across splits; inspect before reporting')


def window_starts(length, window, max_windows=5):
    """Deterministic coverage from beginning to end, without duplicate short windows."""
    if window <= 0 or max_windows < 1:
        raise ValueError('Window sizes must be positive')
    if length <= window:
        return [0]
    n = min(max_windows, int(np.ceil(length / window)))
    return np.unique(np.linspace(0, length-window, n).astype(int)).tolist()

In [13]:
training = (bundle_training if bundle_training is not None else
            prepare_manifest(TRAIN_AUDIO, TRAIN_LABEL, CACHE_ROOT/'train',
                             workers=4, limit=SMOKE_CALLS))
fit, dev = split_training(training, seed=CFG.seed)
assert_disjoint(fit, dev)
summary = pd.DataFrame([
    {'split': name, 'calls': len(frame), 'F': int((frame.y==0).sum()),
     'M': int((frame.y==1).sum()), 'usable': int(frame.usable.sum()),
     'median_seconds': frame.seconds.median()}
    for name, frame in [('fit', fit), ('internal_dev', dev)]])
display(summary)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
summary.set_index('split')[['F', 'M']].plot.bar(ax=axes[0], rot=0)
axes[0].set(title='Class distribution', ylabel='Calls')
axes[1].hist(training.seconds, bins=40, color='#3789a8')
axes[1].set(title='Caller duration', xlabel='Seconds', ylabel='Calls')
fig.tight_layout()
plt.show()

Caller audio:   0%|          | 0/29200 [00:00<?, ?it/s]

,split,calls,F,M,usable,median_seconds
0,fit,23360,12395,10965,23360,27.2605
1,internal_dev,5840,3147,2693,5840,27.0770


## 5. Dataset와 모델

실제 음성을 정규화한 뒤 padding합니다. 학습에서만 무작위 crop·약한 합성 잡음을 사용하고 음높이는
바꾸지 않습니다. 아래 클래스 정의 뒤 학습 셀을 실행할 때 공개 WavLM 가중치를 다운로드합니다.

In [14]:
class AudioDataset(Dataset):
    def __init__(self, frame, cfg, training=False):
        self.frame = frame.reset_index(drop=True)
        self.window = int(cfg.crop_seconds * SR)
        self.training = training
        self.items = []
        for i, row in self.frame.iterrows():
            if not row.usable:
                continue
            source_sr = int(row.get('source_sr', SR))
            n = round(len(np.load(row.path, mmap_mode='r')) * SR / source_sr)
            starts = [None] if training else window_starts(n, self.window, cfg.eval_windows)
            self.items.extend((i, start) for start in starts)

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        row_idx, start = self.items[idx]
        row = self.frame.iloc[row_idx]
        audio = np.load(row.path, mmap_mode='r')
        source_sr = int(row.get('source_sr', SR))
        if start is None:
            n = round(len(audio) * SR / source_sr)
            start = int(np.random.randint(max(1, n-self.window+1)))
        first = int(start * source_sr / SR)
        last = int((start+self.window) * source_sr / SR)
        x = np.array(audio[first:last], dtype=np.float32, copy=True)
        if np.issubdtype(audio.dtype, np.integer):
            x /= 32768.
        if source_sr != SR:
            from scipy.signal import resample_poly
            gcd = math.gcd(source_sr, SR)
            x = resample_poly(x, SR//gcd, source_sr//gcd).astype(np.float32)[:self.window]
        # Noise made locally from RNG; no external training audio, no pitch shifts.
        if self.training and np.random.random() < .5:
            rms = np.sqrt(np.mean(x*x) + 1e-12)
            scale = rms * 10 ** (-np.random.uniform(20, 35) / 20)
            x += np.random.normal(0, scale, len(x)).astype(np.float32)
        # Normalize real samples BEFORE padding, so padding does not change statistics.
        x = (x-x.mean()) / np.sqrt(x.var()+1e-7)
        return x, int(row.y), row_idx


def collate_audio(items):
    n = max(len(item[0]) for item in items)
    values = torch.zeros(len(items), n)
    mask = torch.zeros(len(items), n, dtype=torch.long)
    for i, (audio, _, _) in enumerate(items):
        values[i, :len(audio)] = torch.from_numpy(audio)
        mask[i, :len(audio)] = 1
    return values, mask, torch.tensor([x[1] for x in items]), torch.tensor([x[2] for x in items])


def make_loader(frame, cfg, training=False):
    return DataLoader(AudioDataset(frame, cfg, training),
                      batch_size=cfg.batch_size if training else cfg.eval_batch_size,
                      shuffle=training, num_workers=cfg.num_workers,
                      collate_fn=collate_audio, pin_memory=choose_device(cfg.device).type == 'cuda')

모델의 여러 층을 학습 가능한 가중치로 섞고 유효 프레임만 대상으로 attention을 계산합니다.
가중 평균과 표준편차를 연결하여 F/M을 분류합니다.

In [15]:
class GenderModel(nn.Module):
    def __init__(self, config, pretrained=None):
        super().__init__()
        config = WavLMConfig.from_dict(config) if isinstance(config, dict) else config
        self.backbone = (WavLMModel.from_pretrained(pretrained, config=config,
                                                  revision=getattr(config, '_commit_hash', None))
                         if pretrained else WavLMModel(config))
        size = config.hidden_size
        self.layer_weights = nn.Parameter(torch.zeros(config.num_hidden_layers + 1))
        self.attention = nn.Sequential(nn.Linear(size, 128), nn.Tanh(), nn.Linear(128, 1))
        self.classifier = nn.Sequential(nn.LayerNorm(2*size), nn.Dropout(.2),
                                        nn.Linear(2*size, 128), nn.GELU(),
                                        nn.Dropout(.2), nn.Linear(128, 2))

    def forward(self, input_values, attention_mask):
        states = self.backbone(input_values, attention_mask=attention_mask,
                               output_hidden_states=True).hidden_states
        # Sum avoids allocating an additional [layers,batch,time,hidden] tensor.
        x = sum(w * h for w, h in zip(self.layer_weights.softmax(0), states))
        mask = self.backbone._get_feature_vector_attention_mask(x.shape[1], attention_mask)
        score = self.attention(x).squeeze(-1).float().masked_fill(~mask, float('-inf'))
        weights = score.softmax(-1).unsqueeze(-1)
        x = x.float()
        mean = (weights * x).sum(1)
        var = (weights * (x-mean[:, None]).square()).sum(1)
        return self.classifier(torch.cat((mean, var.clamp_min(1e-5).sqrt()), -1))

    def set_train_stage(self, head_only):
        for p in self.backbone.parameters():
            p.requires_grad_(not head_only)
        self.backbone.freeze_feature_encoder()
        if head_only:
            self.backbone.eval()


def load_checkpoint(path, device):
    payload = torch.load(path, map_location='cpu', weights_only=True)
    model = GenderModel(payload['backbone_config'])
    model.load_state_dict(payload['state_dict'], strict=True)
    return model.to(device).eval(), payload

## 6. 학습·평가 함수

epoch별 내부 dev **통화 단위 Accuracy**로 best 모델을 고릅니다. 같은 내부 dev에서 임계값도 선택하므로
내부 dev 수치는 낙관적일 수 있습니다. 모델 선택에 쓰지 않은 Validation 점수는 별도로 확인합니다.

In [16]:
@torch.inference_mode()
def predict(model, frame, cfg, prior=.5):
    """Average window probabilities, retaining every call and train-only empty fallback."""
    device = next(model.parameters()).device
    model.eval()
    sums, counts = np.zeros(len(frame)), np.zeros(len(frame), dtype=int)
    for x, mask, _, indices in tqdm(make_loader(frame, cfg), desc='Predict', leave=False):
        with amp_context(device):
            logits = model(x.to(device), mask.to(device))
        p = logits.float().softmax(-1)[:, 1].cpu().numpy()
        np.add.at(sums, indices.numpy(), p)
        np.add.at(counts, indices.numpy(), 1)
    return np.divide(sums, counts, out=np.full(len(frame), prior, dtype=float), where=counts > 0)


def metrics(y, probabilities, threshold=.5):
    y = np.asarray(y)
    pred = (np.asarray(probabilities) >= threshold).astype(int)
    return dict(accuracy=float(accuracy_score(y, pred)),
                macro_f1=float(f1_score(y, pred, average='macro', zero_division=0)),
                auc=float(roc_auc_score(y, probabilities)) if len(np.unique(y)) == 2 else None,
                confusion_matrix=confusion_matrix(y, pred, labels=[0, 1]).tolist(),
                calls=len(y), threshold=float(threshold))


def tune_threshold(y, probabilities):
    grid = np.linspace(.35, .65, 61)
    return float(max(grid, key=lambda t: (accuracy_score(y, probabilities >= t), -abs(t-.5))))


def write_json(path, value):
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')


def atomic_save(payload, path):
    path = Path(path)
    temporary = path.with_suffix('.tmp')
    torch.save(payload, temporary)
    temporary.replace(path)

In [17]:
def snapshot_run(run_dir, cfg):
    # 노트북만으로 동작: 저장소 경로나 코드 파일 위치가 필요하지 않습니다.
    packages = {name: importlib.metadata.version(name) for name in
                ('torch', 'transformers', 'numpy', 'scipy', 'pandas', 'scikit-learn')}
    device = choose_device(cfg.device)
    write_json(Path(run_dir)/'provenance.json', {
        'notebook': 'm1_v2.ipynb', 'python': sys.version, 'packages': packages,
        'device': str(device),
        'gpu': torch.cuda.get_device_name(device) if device.type == 'cuda' else None,
        'note': 'Save this notebook with outputs to preserve executed code and results.'})

In [18]:
def train(fit, dev, run_dir, cfg=None, *, resume=False, model=None):
    cfg = cfg or TrainConfig()
    if min(cfg.epochs, cfg.batch_size, cfg.accumulation, cfg.eval_batch_size, cfg.eval_windows) < 1:
        raise ValueError('Epoch/batch/window counts must be positive')
    if cfg.crop_seconds < .25 or not 0 <= cfg.head_epochs < cfg.epochs:
        raise ValueError('Require crop >= .25 s and 0 <= head_epochs < epochs')
    run_dir = Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
    if (run_dir/'config.json').exists() and not resume:
        raise FileExistsError('Use a fresh RUN_DIR or resume=True for epoch-boundary restart')
    if not fit.usable.any() or not dev.usable.any():
        raise ValueError('Need usable caller audio in both fit and dev')
    if set(fit.stem) & set(dev.stem):
        raise ValueError('Fit and dev calls overlap')
    if set(fit.loc[fit.usable, 'audio_hash']) & set(dev.loc[dev.usable, 'audio_hash']):
        raise ValueError('Fit and dev audio hashes overlap')
    split_hash = hashlib.sha256(pd.concat([fit.assign(part='fit'), dev.assign(part='dev')])[
        ['stem', 'audio_hash', 'y', 'part']].to_csv(index=False).encode()).hexdigest()
    seed_everything(cfg.seed)
    device = choose_device(cfg.device)
    if device.type == 'cpu':
        print('CPU detected: full WavLM training can be slow. CUDA is used automatically when available.')
    prior = float(fit.y.mean())
    last = None
    if resume:
        last = torch.load(run_dir/'last.pt', map_location='cpu', weights_only=True)
        if last['train_config'] != asdict(cfg) or last['split_hash'] != split_hash:
            raise ValueError('Resume config/data mismatch')
        model = GenderModel(last['backbone_config'])
        model.load_state_dict(last['state_dict'])
    elif model is None:
        backbone_cfg = WavLMConfig.from_pretrained(cfg.model_name)
        backbone_cfg.layerdrop = 0.
        backbone_cfg.mask_time_prob = .04
        backbone_cfg.mask_time_length = 2
        backbone_cfg.mask_time_min_masks = 0
        model = GenderModel(backbone_cfg, pretrained=cfg.model_name)
    model = model.to(device)
    model.set_train_stage(False)
    if cfg.gradient_checkpointing:
        model.backbone.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
    backbone, head = [], []
    for name, p in model.named_parameters():
        if p.requires_grad:
            (backbone if name.startswith('backbone.') else head).append(p)
    optimizer = torch.optim.AdamW([{'params': backbone, 'lr': cfg.backbone_lr},
                                  {'params': head, 'lr': cfg.head_lr}], weight_decay=cfg.weight_decay)
    loader = make_loader(fit, cfg, training=True)
    steps = math.ceil(len(loader)/cfg.accumulation) * cfg.epochs
    scheduler = get_cosine_schedule_with_warmup(optimizer, max(1, int(.1*steps)), steps)
    scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda' and not torch.cuda.is_bf16_supported())
    best_acc, bad_epochs, start_epoch = -1., 0, 0
    if last:
        optimizer.load_state_dict(last['optimizer'])
        scheduler.load_state_dict(last['scheduler'])
        scaler.load_state_dict(last['scaler'])
        best_acc, bad_epochs, start_epoch = last['best_acc'], last['bad_epochs'], last['epoch']+1
    else:
        write_json(run_dir/'config.json', asdict(cfg))
        snapshot_run(run_dir, cfg)
        fit.to_csv(run_dir/'fit_manifest.csv', index=False)
        dev.to_csv(run_dir/'dev_manifest.csv', index=False)
    for epoch in range(start_epoch, cfg.epochs):
        if bad_epochs >= cfg.patience:
            break
        # Reseeding per epoch makes last.pt resume independent of previous RNG consumption.
        seed_everything(cfg.seed + epoch)
        model.train()
        model.set_train_stage(epoch < cfg.head_epochs)
        optimizer.zero_grad(set_to_none=True)
        total_loss, seen = 0., 0
        for step, (x, mask, y, _) in enumerate(tqdm(loader, desc=f'Epoch {epoch+1}/{cfg.epochs}')):
            group_start = (step // cfg.accumulation) * cfg.accumulation
            # Normalize by actual examples, including the final partial batch/group.
            group_end = min(group_start + cfg.accumulation, len(loader))
            group_samples = min(group_end*cfg.batch_size, len(loader.dataset)) - group_start*cfg.batch_size
            with amp_context(device):
                logits = model(x.to(device), mask.to(device))
                loss = torch.nn.functional.cross_entropy(logits.float(), y.to(device), reduction='sum')
            scaler.scale(loss/group_samples).backward()
            total_loss += float(loss.detach())
            seen += len(y)
            if (step+1) % cfg.accumulation == 0 or step+1 == len(loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.)
                old_scale = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                if scaler.get_scale() >= old_scale:
                    scheduler.step()
                optimizer.zero_grad(set_to_none=True)
        p = predict(model, dev, cfg, prior)
        report = metrics(dev.y, p)
        report.update(epoch=epoch+1, train_loss=total_loss/max(seen, 1))
        print(json.dumps(report), flush=True)
        with (run_dir/'history.jsonl').open('a', encoding='utf-8') as f:
            f.write(json.dumps(report)+'\n')
        improved = report['accuracy'] > best_acc
        best_acc = max(best_acc, report['accuracy'])
        bad_epochs = 0 if improved else bad_epochs+1
        payload = dict(state_dict={k: v.detach().cpu() for k, v in model.state_dict().items()},
                       backbone_config=model.backbone.config.to_dict(), train_config=asdict(cfg),
                       preprocessing_version=PREPROCESS_VERSION, label_map={'F': 0, 'M': 1},
                       threshold=.5, prior=prior, epoch=epoch, best_acc=best_acc,
                       split_hash=split_hash, bad_epochs=bad_epochs)
        if improved:
            atomic_save(payload, run_dir/'best.pt')
        atomic_save(dict(payload, optimizer=optimizer.state_dict(), scheduler=scheduler.state_dict(),
                         scaler=scaler.state_dict()), run_dir/'last.pt')
    # Reuse the same allocation to avoid keeping two full models in GPU memory.
    best = torch.load(run_dir/'best.pt', map_location='cpu', weights_only=True)
    model.load_state_dict(best['state_dict'])
    p = predict(model, dev, cfg, prior)
    threshold = tune_threshold(dev.y.to_numpy(), p)
    best['threshold'] = threshold
    atomic_save(best, run_dir/'best.pt')
    report = metrics(dev.y, p, threshold)
    report['uncalibrated_accuracy'] = metrics(dev.y, p)['accuracy']
    write_json(run_dir/'dev_metrics.json', report)
    dev[['stem', 'y', 'seconds', 'usable']].assign(p_male=p, prediction=(p >= threshold).astype(int)).to_csv(
        run_dir/'dev_predictions.csv', index=False)
    return run_dir/'best.pt'

## 7. 학습 실행과 학습 곡선

진행률, loss·Accuracy가 출력됩니다. 최적 모델은 `best.pt`, 재개 상태는 `last.pt`입니다.
새 실험은 새 RUN_DIR를 사용하세요. 중단 재개는 `RESUME_RUN_DIR`를 지정하고 동일 데이터·설정으로 실행합니다.

In [19]:
CHECKPOINT = train(fit, dev, RUN_DIR, CFG, resume=RESUME_RUN_DIR is not None)
history = pd.read_json(RUN_DIR/'history.jsonl', lines=True)
display(history[['epoch', 'train_loss', 'accuracy', 'macro_f1', 'auc']])
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(history.epoch, history.train_loss, marker='o', color='#3789a8')
axes[0].set(xlabel='Epoch', ylabel='Cross entropy', title='Training loss')
axes[1].plot(history.epoch, history.accuracy, marker='o', color='#d77e35')
axes[1].set(xlabel='Epoch', ylabel='Accuracy', title='Internal dev accuracy')
for ax in axes:
    ax.grid(alpha=.2)
fig.tight_layout()
fig.savefig(RUN_DIR/'learning_curve.png', dpi=160, bbox_inches='tight')
plt.show()
print('Best checkpoint:', CHECKPOINT)

Epoch 1/8:   0%|          | 0/5840 [00:00<?, ?it/s]

/home/a202355692/micromamba/envs/aienv/lib/python3.10/site-packages/torch/nn/functional.py:5962: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


Predict:   0%|          | 0/3227 [00:00<?, ?it/s]

{"accuracy": 0.9717465753424658, "macro_f1": 0.9715251382409327, "auc": 0.994493131517872, "confusion_matrix": [[3095, 52], [113, 2580]], "calls": 5840, "threshold": 0.5, "epoch": 1, "train_loss": 0.45137972234435736}


Epoch 2/8:   0%|          | 0/5840 [00:00<?, ?it/s]

...

{"accuracy": 0.9881849315068493, "macro_f1": 0.9881108470552351, "auc": 0.9975039148088507, "confusion_matrix": [[3116, 31], [38, 2655]], "calls": 5840, "threshold": 0.5, "epoch": 7, "train_loss": 0.09272719970178689}


Predict:   0%|          | 0/3227 [00:00<?, ?it/s]

/home/a202355692/micromamba/envs/aienv/lib/python3.10/site-packages/torch/nn/functional.py:5962: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


,epoch,train_loss,accuracy,macro_f1,auc
0,1,0.451380,0.971747,0.971525,0.994493
1,2,0.139821,0.986301,0.986223,0.997238
2,3,0.112867,0.984589,0.984478,0.997209
3,4,0.106280,0.988356,0.988288,0.997495
4,5,0.099239,0.987671,0.987600,0.997523
5,6,0.092567,0.987500,0.987422,0.997424
6,7,0.092727,0.988185,0.988111,0.997504


Best checkpoint: /home/a202355692/data/runs/m1_v2/20260919_165146_wavlm_experiment/best.pt


## 8. 시각화 코드

혼동행렬의 **행은 실제 정답, 열은 예측**입니다. 왼쪽은 통화 수, 오른쪽은 실제 성별별 비율입니다.
F→M, M→F 중 어느 오류가 많은지 확인하세요.

다음 함수는 점수 표, F/M별 precision·recall·F1, 혼동행렬 2개, ROC/PR 곡선,
성별별 예측 확률 분포, 음성 길이별 정확도·표본 수, 오분류 목록을 출력하고 그림으로 저장합니다.

In [20]:
def show_results(predictions, report, title, output_dir):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    y = predictions.y.to_numpy(dtype=int)
    pred = predictions.prediction.to_numpy(dtype=int)
    p = predictions.p_male.to_numpy(dtype=float)
    display(pd.Series({k: report.get(k) for k in
                       ('accuracy', 'macro_f1', 'auc', 'calls', 'threshold')}, name=title))
    class_scores = classification_report(
        y, pred, labels=[0, 1], target_names=['F', 'M'], output_dict=True, zero_division=0)
    per_class = pd.DataFrame({key: value for key, value in class_scores.items()
                             if isinstance(value, dict)}).T
    display(per_class)
    per_class.to_csv(output_dir/'classification_report.csv')
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
    for ax, normalization, fmt, label in [
        (axes[0], None, 'd', 'Counts'), (axes[1], 'true', '.1%', 'Row-normalized')]:
        ConfusionMatrixDisplay.from_predictions(y, pred, labels=[0, 1], display_labels=['F', 'M'],
            normalize=normalization, values_format=fmt, cmap='Blues', colorbar=False, ax=ax)
        ax.set_title(label)
    fig.suptitle(f"{title} | accuracy = {report['accuracy']:.4f}")
    fig.tight_layout()
    fig.savefig(output_dir/'confusion_matrix.png', dpi=170, bbox_inches='tight')
    plt.show()

    fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
    if len(np.unique(y)) == 2:
        fpr, tpr, _ = roc_curve(y, p)
        precision, recall, _ = precision_recall_curve(y, p)
        axes[0].plot(fpr, tpr, label=f"AUC={roc_auc_score(y, p):.4f}")
        axes[0].plot([0, 1], [0, 1], '--', color='gray')
        axes[1].plot(recall, precision, label=f"AP={average_precision_score(y, p):.4f}")
        axes[1].axhline(y.mean(), linestyle='--', color='gray', label='M prevalence')
        axes[0].legend()
        axes[1].legend()
    else:
        for ax in axes[:2]:
            ax.text(.5, .5, 'Requires both classes', ha='center', transform=ax.transAxes)
    axes[0].set(title='ROC (M positive)', xlabel='False positive rate', ylabel='True positive rate')
    axes[1].set(title='Precision-recall (M positive)', xlabel='Recall', ylabel='Precision')
    for label, name, color in [(0, 'True F', '#3789a8'), (1, 'True M', '#d77e35')]:
        axes[2].hist(p[y == label], bins=np.linspace(0, 1, 26), alpha=.6, label=name, color=color)
    axes[2].axvline(report['threshold'], color='black', linestyle='--', label='Decision threshold')
    axes[2].set(title='Predicted probability', xlabel='P(M)', ylabel='Calls')
    axes[2].legend()
    fig.tight_layout()
    fig.savefig(output_dir/'roc_pr_probability.png', dpi=170, bbox_inches='tight')
    plt.show()

    diagnostic = predictions.copy()
    diagnostic['duration_bin'] = pd.cut(diagnostic.seconds, [-1, .25, 2, 6, 20, np.inf],
        labels=['<=0.25s', '0.25-2s', '2-6s', '6-20s', '>20s'])
    diagnostic['correct'] = diagnostic.y == diagnostic.prediction
    duration = diagnostic.groupby('duration_bin', observed=True).agg(
        calls=('correct', 'size'), accuracy=('correct', 'mean'))
    display(duration)
    fig, ax = plt.subplots(figsize=(8, 3.5))
    ax.bar(duration.index.astype(str), duration.accuracy, color='#3789a8')
    ax.set(ylim=(0, 1.12), xlabel='Caller duration', ylabel='Accuracy', title='Accuracy by audio duration')
    for i, row in enumerate(duration.itertuples()):
        ax.text(i, row.accuracy+.02, f'n={row.calls}', ha='center')
    fig.tight_layout()
    fig.savefig(output_dir/'accuracy_by_duration.png', dpi=170, bbox_inches='tight')
    plt.show()
    duration.to_csv(output_dir/'duration_report.csv')
    errors = diagnostic.loc[~diagnostic.correct,
        ['stem', 'y', 'prediction', 'p_male', 'seconds', 'usable']].copy()
    errors['confidence'] = np.where(errors.prediction == 1, errors.p_male, 1-errors.p_male)
    errors = errors.sort_values('confidence', ascending=False)
    errors.to_csv(output_dir/'errors.csv', index=False)
    print(f'Errors: {len(errors)}/{len(predictions)} | y/prediction: 0=F, 1=M')
    display(errors.head(20))
    return {'classification': per_class, 'duration': duration, 'errors': errors}

## 9. 내부 dev 결과 확인

학습 때 저장한 예측을 읽어 그립니다. 결과만 다시 보고 싶으면 패키지·함수 정의 셀을 실행하고
`RUN_DIR`를 이전 실험 폴더로 지정하세요. 학습 셀을 다시 실행할 필요가 없습니다.

In [21]:
# 결과만 다시 확인할 때: RUN_DIR = Path('이전 실험 폴더')
dev_report = json.loads((RUN_DIR/'dev_metrics.json').read_text(encoding='utf-8'))
dev_predictions = pd.read_csv(RUN_DIR/'dev_predictions.csv')
dev_analysis = show_results(dev_predictions, dev_report, 'Internal dev', RUN_DIR/'figures/dev')

accuracy        0.988356
macro_f1        0.988288
auc             0.997495
calls        5840.000000
threshold       0.500000
Name: Internal dev, dtype: float64

,precision,recall,f1-score,support
F,0.990443,0.987925,0.989182,3147.0
M,0.985931,0.988860,0.987393,2693.0
macro avg,0.988187,0.988393,0.988288,5840.0
weighted avg,0.988362,0.988356,0.988357,5840.0


,calls,accuracy
duration_bin,,
0.25-2s,1,1.000000
2-6s,26,0.923077
6-20s,1336,0.991018
>20s,4477,0.987938


Errors: 68/5840 | y/prediction: 0=F, 1=M


,stem,y,prediction,p_male,seconds,usable,confidence
4232,651e51d797589376a420b17c_20221201,0,1,0.997285,3.692,True,0.997285
218,651e496f5d60e22224166d36_20220106,0,1,0.996225,42.826,True,0.996225
4919,651e545d85c12f9c1b91f3b7_20220303,1,0,0.004711,37.083,True,0.995289
1749,651e4bfaf12d88de38d4981c_20220507,0,1,0.995271,41.338,True,0.995271
2453,651e4e89429d02dab45a2b09_20220707,0,1,0.995035,24.299,True,0.995035
4650,651e539031ce2800caf02b8c_20220104,1,0,0.007077,26.709,True,0.992923
3597,651e5013d7e77fe70c0b0226_20221002,0,1,0.991631,21.209,True,0.991631
2490,651e4e89429d02dab45a2d0e_20220707,0,1,0.986419,24.422,True,0.986419
1885,651e4c5b48bae90750379921_20220602,0,1,0.983386,26.359,True,0.983386
5838,651e5dd6049c168a19187cd4_20221105,1,0,0.017309,5.458,True,0.982691


## 10. 공식 Validation 결과와 혼동행렬

선택한 모델·임계값을 그대로 사용합니다. 결과를 보고 반복해서 설정을 고르면 독립적인 최종 평가가
아니므로 실험 이력에 구분해서 기록하세요. `SMOKE_CALLS`를 지정한 실행에서는 이 평가를 건너뜁니다.

In [22]:
def evaluate_experiment(checkpoint, frame, output_dir, device='auto'):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    model, saved = load_checkpoint(checkpoint, choose_device(device))
    if saved['preprocessing_version'] != PREPROCESS_VERSION:
        raise ValueError('Preprocessing version mismatch')
    cfg = TrainConfig(**saved['train_config'])
    cfg.device = device
    p = predict(model, frame, cfg, saved['prior'])
    report = metrics(frame.y, p, saved['threshold'])
    report['empty_or_unusable_calls'] = int((~frame.usable).sum())
    predictions = frame[['stem', 'y', 'seconds', 'usable']].copy()
    predictions['p_male'] = p
    predictions['prediction'] = (p >= saved['threshold']).astype(int)
    write_json(output_dir/'metrics.json', report)
    predictions.to_csv(output_dir/'predictions.csv', index=False)
    return report, predictions

In [23]:
if RUN_VALIDATION and SMOKE_CALLS is None:
    if bundle_validation is not None:
        validation = bundle_validation
    else:
        for path in (VAL_AUDIO, VAL_LABEL):
            if not path.is_dir():
                raise FileNotFoundError(path)
        validation = prepare_manifest(VAL_AUDIO, VAL_LABEL, CACHE_ROOT/'validation', workers=4)
    assert_disjoint(training, validation)
    validation_report, validation_predictions = evaluate_experiment(
        CHECKPOINT, validation, RUN_DIR/'validation', device=CFG.device)
    validation_analysis = show_results(validation_predictions, validation_report,
                                      'Official validation', RUN_DIR/'figures/validation')
    comparison = pd.DataFrame([
        {'model': 'Historical MFCC/F0 + LogisticRegression', 'accuracy': 0.9434065934065934},
        {'model': 'WavLM v2 (this run)', 'accuracy': validation_report['accuracy']}])
    display(comparison)
    comparison.to_csv(RUN_DIR/'comparison.csv', index=False)
    print('Difference from historical baseline (percentage points):',
          round(100*(validation_report['accuracy']-0.9434065934065934), 3))
else:
    print('Validation skipped. Internal dev results are shown above.')

Caller audio:   0%|          | 0/3640 [00:00<?, ?it/s]

Predict:   0%|          | 0/2012 [00:00<?, ?it/s]

/home/a202355692/micromamba/envs/aienv/lib/python3.10/site-packages/torch/nn/functional.py:5962: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


accuracy        0.987363
macro_f1        0.987292
auc             0.996465
calls        3640.000000
threshold       0.500000
Name: Official validation, dtype: float64

,precision,recall,f1-score,support
F,0.989759,0.986728,0.988241,1959.0
M,0.984588,0.988102,0.986342,1681.0
macro avg,0.987174,0.987415,0.987292,3640.0
weighted avg,0.987371,0.987363,0.987364,3640.0


,calls,accuracy
duration_bin,,
0.25-2s,2,1.000000
2-6s,22,1.000000
6-20s,812,0.987685
>20s,2804,0.987161


Errors: 46/3640 | y/prediction: 0=F, 1=M


,stem,y,prediction,p_male,seconds,usable,confidence
2368,651e50c72f06ed4a6e31e2b2_20221006,1,0,0.003075,33.729,True,0.996925
3106,651e54776f6c84c1ff30cc90_20220304,1,0,0.004155,51.974,True,0.995845
810,651e4b6d304f68a2e7021a9e_20220503,1,0,0.006412,23.371,True,0.993588
2240,651e5013d7e77fe70c0b0124_20221002,0,1,0.993140,28.337,True,0.993140
547,651e4a554767d37510dad9f6_20220401,1,0,0.006927,49.153,True,0.993073
1228,651e4c8ad24efd3f59d0158d_20220607,0,1,0.989121,30.084,True,0.989121
3410,651e5500835c8a556d7e2c85_20220404,0,1,0.988567,81.698,True,0.988567
767,651e4b1829d82455557a2187_20220502,1,0,0.014239,32.768,True,0.985761
1439,651e4e712e98ee7120968eb5_20220704,0,1,0.983781,28.443,True,0.983781
1176,651e4c754428de718c773c45_20220606,1,0,0.016525,35.326,True,0.983475


,model,accuracy
0,Historical MFCC/F0 + LogisticRegression,0.943407
1,WavLM v2 (this run),0.987363


Difference from historical baseline (percentage points): 4.396


## 11. 저장과 다시 보기

**출력이 포함된 노트북을 저장하세요.** Colab에서 ipynb를 다운로드하면 표와 그림도 함께 보관됩니다.
`RUN_DIR`의 `history.jsonl`, `learning_curve.png`에는 학습 추이,
`dev_metrics.json`, `dev_predictions.csv`에는 내부 dev 결과,
`validation/`에는 공식 평가 결과, `figures/dev`, `figures/validation`에는 분석 그림이 남습니다.
`best.pt`, `last.pt`는 실험 모델 보관·학습 재개용입니다. 마지막 셀은 결과 요약 이미지를 만들고,
로컬 재현 안내·환경 정보·모든 산출물의 SHA-256 체크섬도 파일로 저장합니다.

WavLM의 영어 사전학습과 한국어 전화 음성 사이에 차이가 있으므로 실제 성능을 검증해야 합니다.
과거 baseline은 이번 내부 분할과 완전히 동일한 통제 실험이 아닙니다.

참고: [지정 GitHub](https://github.com/pnu-final-boss/2026-data-creator-camp),
[WavLM 모델 카드](https://huggingface.co/microsoft/wavlm-base-plus),
[WavLM 논문](https://arxiv.org/abs/2110.13900). 문제 PDF 5·7쪽의 조건을 기준으로 구성했습니다.

In [24]:
def save_result_artifacts(run_dir):
    run_dir = Path(run_dir)
    reports = []
    for label, path in [('Internal dev', run_dir/'dev_metrics.json'),
                        ('Official validation', run_dir/'validation/metrics.json')]:
        if path.is_file():
            report = json.loads(path.read_text(encoding='utf-8'))
            reports.append([label, report.get('calls'), report.get('accuracy'),
                            report.get('macro_f1'), report.get('auc'), report.get('threshold')])

    fig = plt.figure(figsize=(12, 8), constrained_layout=True)
    grid = fig.add_gridspec(2, 1, height_ratios=[1, 2])
    table_ax = fig.add_subplot(grid[0])
    table_ax.axis('off')
    table_ax.set_title('M1 WavLM experiment summary', fontsize=16, pad=14)
    if reports:
        formatted = [[row[0], str(row[1]),
                      *[('N/A' if value is None else f'{value:.4f}') for value in row[2:]]]
                     for row in reports]
        table = table_ax.table(cellText=formatted,
            colLabels=['Dataset', 'Calls', 'Accuracy', 'Macro F1', 'AUC', 'Threshold'],
            cellLoc='center', loc='center')
        table.auto_set_font_size(False)
        table.set_fontsize(10)
        table.scale(1, 1.6)
    else:
        table_ax.text(.5, .5, 'No completed metrics found', ha='center', va='center')

    history_ax = fig.add_subplot(grid[1])
    history_path = run_dir/'history.jsonl'
    if history_path.is_file():
        history = pd.read_json(history_path, lines=True)
        history_ax.plot(history.epoch, history.accuracy, marker='o', label='Dev accuracy')
        if 'macro_f1' in history:
            history_ax.plot(history.epoch, history.macro_f1, marker='s', label='Dev macro F1')
        history_ax.set(xlabel='Epoch', ylabel='Score', ylim=(0, 1.02), title='Training history')
        history_ax.grid(alpha=.2)
        history_ax.legend()
    else:
        history_ax.axis('off')
        history_ax.text(.5, .5, 'No training history found', ha='center', va='center')
    summary_path = run_dir/'result_summary.png'
    fig.savefig(summary_path, dpi=180, bbox_inches='tight')
    plt.close(fig)

    environment = {
        'python': sys.version,
        'packages': {name: importlib.metadata.version(name) for name in
                     ('torch', 'transformers', 'numpy', 'scipy', 'pandas', 'scikit-learn', 'matplotlib')},
        'cuda_build': torch.version.cuda,
        'cuda_available': torch.cuda.is_available(),
        'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    }
    write_json(run_dir/'environment.json', environment)
    (run_dir/'LOCAL_REPRODUCTION.md').write_text(
        '# Local reproduction\n\n'
        'Keep this entire run directory together with `m1_v2_linux_r5.ipynb`.\n'
        '`best.pt` contains the complete trained model weights, backbone configuration, '
        'decision threshold, label map, and training configuration. It does not require '
        'downloading the original Hugging Face checkpoint for inference.\n\n'
        'The saved PNG, CSV, and JSON files reproduce the reported result without the raw data. '
        'Re-running evaluation requires the original Validation WAV/JSON data. '
        'Re-running training requires all Training/Validation data and the package versions in '
        '`environment.json`. Exact bitwise retraining across different GPU/CUDA versions is not guaranteed.\n',
        encoding='utf-8')

    manifest = []
    for path in sorted(run_dir.rglob('*')):
        if not path.is_file() or path.name == 'artifact_manifest.json':
            continue
        digest = hashlib.sha256()
        with path.open('rb') as stream:
            for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
                digest.update(chunk)
        manifest.append({'path': str(path.relative_to(run_dir)),
                         'bytes': path.stat().st_size, 'sha256': digest.hexdigest()})
    write_json(run_dir/'artifact_manifest.json', manifest)
    return summary_path, sorted(path for path in run_dir.rglob('*.png'))

SUMMARY_IMAGE, RESULT_IMAGES = save_result_artifacts(RUN_DIR)
print('Experiment results:', RUN_DIR)
print('Best checkpoint:', RUN_DIR/'best.pt')
print('Summary image:', SUMMARY_IMAGE)
print('Saved result images:')
for image_path in RESULT_IMAGES:
    print(' -', image_path.relative_to(RUN_DIR))
print('Artifact checksums:', RUN_DIR/'artifact_manifest.json')
print('노트북을 실행 출력과 함께 저장하세요.')

Experiment results: /home/a202355692/data/runs/m1_v2/20260919_165146_wavlm_experiment
Best checkpoint: /home/a202355692/data/runs/m1_v2/20260919_165146_wavlm_experiment/best.pt
Summary image: /home/a202355692/data/runs/m1_v2/20260919_165146_wavlm_experiment/result_summary.png
Saved result images:
 - figures/dev/accuracy_by_duration.png
 - figures/dev/confusion_matrix.png
 - figures/dev/roc_pr_probability.png
 - figures/validation/accuracy_by_duration.png
 - figures/validation/confusion_matrix.png
 - figures/validation/roc_pr_probability.png
 - learning_curve.png
 - result_summary.png
Artifact checksums: /home/a202355692/data/runs/m1_v2/20260919_165146_wavlm_experiment/artifact_manifest.json
노트북을 실행 출력과 함께 저장하세요.
